# Demostración del funcionamiento de la librería para la detección anticipada de riesgos con razonadores sobre flujos

Importamos las librerías necesarias

In [1]:
from social_media.classifier.discourse_element_classifier import DiscourseElementClassifier
from social_media.classifier.prompts import DISCOURSE_ELEMENT_CLASSIFIER_SYSTEM_TEMPLATE

from social_media.erisk.discourse_element_provider import DiscourseElementProvider
from social_media.reasoner.incremental_reasoner import IncrementalReasoner
from social_media.classifier.classifier_codomain import ClassifierCodomain
from social_media.classifier.mock_classifier import MockClassifier

from social_media.domain.discourse_element import DiscourseElement
from social_media.domain.user import User

from llms_kgs.llms import Gemma3_4B, Gemini25Flash

import datetime as dt

import pprint

Instanciamos a un LLM y construimos al clasificador de elementos de discurso. 

In [2]:
gemma = Gemma3_4B(temperature = 0.5, schema = ClassifierCodomain.model_json_schema())
gemini = Gemini25Flash(temperature = 0.5, schema = ClassifierCodomain.model_json_schema())

discourse_element_classifier = DiscourseElementClassifier(llm=gemini,
                                                         system_template=DISCOURSE_ELEMENT_CLASSIFIER_SYSTEM_TEMPLATE)


Instanciamos al razonador sobre flujos, el cual tiene como fecha de referencia el día 1 de julio de 2026

In [3]:
reasoner = IncrementalReasoner(classifier = discourse_element_classifier,
                               reference_date = dt.datetime(year=2026,month=7,day=1), win_size=30)

Creamos los elementos de discurso.

In [4]:
element0 = DiscourseElement(
    content = "I dislike everything about myself and I want to harm my arm",
    creation_date = dt.datetime(year=2026, month=7, day=2),
    discourse_id = "0", 
    sender = User("0"),
    answers_to = None)

element1 = DiscourseElement(
    content = "I am feeling sad and I do not want to get up to eat",
    creation_date = dt.datetime(year=2026, month=7, day=3),
    discourse_id = "1", 
    sender = User("0"),
    answers_to = None)

element2 = DiscourseElement(
    content = "I understand you. My psychologist diagnosed me with depression and I am feeling like a failure",
    creation_date = dt.datetime(year=2026, month=7, day=3),
    discourse_id = "2", 
    sender = User("1"),
    answers_to = element1)

element3 = DiscourseElement(
    content = "Today is a happy day and I want to smile to everyone",
    creation_date = dt.datetime(year=2026, month=7, day=4),
    discourse_id = "3", 
    sender = User("2"),
    answers_to = None)

Solicitamos al razonador que interprete los elementos de discurso. 

In [14]:
reasoner.interpret_discourse_element(element0)
reasoner.interpret_discourse_element(element1)
reasoner.interpret_discourse_element(element2)
reasoner.interpret_discourse_element(element3)
pprint.pprint(reasoner.get_evaluation_function())

{1: ['depressed(0, 0)'], 2: ['depressed(0, 1)', 'depressed(1, 2)']}


Solicitamos al razonador que emita una alarma por los usuarios en riesgo.

In [15]:
alarming_users = reasoner.get_alarming_users(dt.datetime(year=2026,month=7,day=8))

Imprimimos los resultados y las explicaciones para que el experto puede interpretar las alarmas. 

In [17]:
for user, diagnostic_list in alarming_users:
    print("In the last month two messages triggered the alarm for the user with ID " + user.get_id())
    for diagnostic_data in diagnostic_list:
        print("<Message>")
        print(diagnostic_data.discourse_element.get_content())
        print("<Analysis>")
        print(diagnostic_data.classifier_result.explanation)
    print("="*80)

In the last month two messages triggered the alarm for the user with ID 0
<Message>
I dislike everything about myself and I want to harm my arm
<Analysis>
The user explicitly states self-dislike and an intent to self-harm, specifically mentioning 'harm my arm'. These are direct and severe indicators of psychological distress, commonly associated with depressive episodes and a significant risk for self-injurious behavior. The statement reflects feelings of worthlessness and a desire to inflict pain, which are core symptoms of severe depression.
<Message>
I am feeling sad and I do not want to get up to eat
<Analysis>
The user explicitly states feeling sad, which is a core symptom of depression. Additionally, expressing a lack of desire to get up to eat indicates a loss of motivation and potentially a reduced appetite or anhedonia, both of which are common indicators of depressive states. This combination of low mood and lack of motivation for basic self-care activities suggests a positiv

Ahora cargaremos el lote de datos completo y usaremos a un clasificador de juguete para probar el funcionamiento del razonador con un gran volumen de datos. 

In [42]:
discourse_elements = DiscourseElementProvider.transform_dataset_to_discourse_element_list('../mock_dataset')
print(len(discourse_elements))


24439


Imprimimos algunos de los mensajes.

In [43]:
for discourse_element in discourse_elements[0:15]:
    print(discourse_element._creation_date)
    print(discourse_element._content)

2012-12-10 00:00:00
Favourite Che Quote?
I have several. These are my top three:

1. It is better to die standing than to live on your knees.

2. Let me say at the risk of seeming ridiculous that the true revolutionary is guided by great feelings of love.

3.I am not a liberator. Liberators do not exist. The people liberate themselves.

2012-12-11 00:00:00
All fantastic. I think in that first one, he was paraphrasing Zapata, who said something similar, and bringing the quotation back to apply to a new struggle of the same spirit.

**


*If you tremble with indignation at every injustice, then you are a comrade of mine.*

**

*The life of a single human being is worth a million times more than all the property of the richest man on earth.*

**


*I knew that the moment the great governing spirit strikes the blow to divide all humanity into just two opposing factions, I would be on the side of the common people.*
2012-12-11 00:00:00
"Above all, always be capable of feeling deeply any inj

Construimos un clasificador y un razonador de juguete. 

In [44]:
mock_classifier = MockClassifier(0.5)

mock_reasoner = IncrementalReasoner(classifier = mock_classifier, reference_date = dt.datetime(year=2010,month=1,day=19), win_size = 365)

Procesamos todos los elementos de discurso.

In [45]:
for discourse_element in discourse_elements:
    mock_reasoner.interpret_discourse_element(discourse_element)

Evaluamos el flujo respecto al 8 de enero de 2023.

In [46]:
alarming_users = mock_reasoner.get_alarming_users(dt.datetime(year=2020,month=5,day=1))

Imprimimos los resultados.

In [48]:
for user, diagnostic_list in alarming_users[:5]:
    print("In the last 365 days two messages triggered the alarm for the user with ID " + user.get_id())
    for diagnostic_data in diagnostic_list:
        print("<Message>")
        print(diagnostic_data.discourse_element.get_content())
        print("<Analysis>")
        print(diagnostic_data.classifier_result.explanation)
    print("="*80)

In the last 365 days two messages triggered the alarm for the user with ID deleted_user
<Message>
My grandpa once said to me (after I came clean to him about my self harm and thoughts of ending it all) that the only way for life to get better is to let it go on.
I don’t know why, but it really hit something inside me ... 
Life is both the best and the worst I’m going to experience, but now I’m finally at a place where I feel that all the bad things I’ve gone through are okay, because they led me to where I am now. 
I’m becoming a nurse, and I absolutely love the idea of helping other people heal, for the rest of my life. If I hadn’t gone through so much sh*t, I’m not sure that I would have ended up where I am today.

I hope that you’ll find a grandpa/Neil Simon/general good person too! Because when everything clicks in you it’s like coming up for a breath of fresh air!
It won’t be easy, and it won’t be painless - but you suddenly find a strength that you maybe didn’t even know you had.